Step 1: Load and prepare dataset for MLP

In first step, we will load dataset and prepare it for training a MLP using PyTorch. The given dataset has 4000 rows of data with 7 features and 1 class label (as 8th column).




In [1]:
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# we load the dataset from the given CSV data file
data = np.loadtxt("MLoGPU_data3_train.csv", delimiter=",")

# we split features (first 7 columns) and labels (8th / last column).
X = data[:, :-1]
y = data[:, -1].astype(int) - 1  # -1 will Convert labels from 1–7 to 0–6 so PyTorch can use them.

# we split data into training and validation sets.
# 20% of the data will be used for validation.
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

# Then we Standardize the features so that each feature has mean 0 and variance 1.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

# we convert the NumPy arrays into PyTorch tensors
# float32 is the standard type for neural network inputs
X_train_torch = torch.tensor(X_train, dtype=torch.float32)
X_val_torch   = torch.tensor(X_val, dtype=torch.float32)

# PyTorch classification accepts long integers as the Labels should be.
y_train_torch = torch.tensor(y_train, dtype=torch.long)
y_val_torch   = torch.tensor(y_val, dtype=torch.long)

print("Data loaded successfully and prepared with:")
print("Training size:", X_train_torch.shape)
print("Validation size:", X_val_torch.shape)

# we create TensorDataset and DataLoader objects

from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train_torch, y_train_torch)
val_ds   = TensorDataset(X_val_torch, y_val_torch)

# Below are the DataLoaders for batching
train_ldr = DataLoader(train_ds, batch_size=64, shuffle=True)
val_ldr   = DataLoader(val_ds, batch_size=64, shuffle=False)

print("DataLoaders successfully created.")
print("Training batches number:", len(train_ldr))
print("Validation batches number:", len(val_ldr))

Data loaded successfully and prepared with:
Training size: torch.Size([3200, 7])
Validation size: torch.Size([800, 7])
DataLoaders successfully created.
Training batches number: 50
Validation batches number: 13


Step 2: Defining a MLP Model Using PyTorch

In second step, we define a MLP (multilayer perception) for this classificaiton task. The model will use PyTorch's class which will allow us to construct neural network. We have to compare CPU vs GPU performance thus the architecture we will use, will have 1 hidden layer with a non-linear activation function. The output will also have 7 unit as we have 7 classes.

In [2]:
import torch.nn as nn

# Defining a small MLP model with:
# - an input layer with 7 features,
# - one hidden layer with 32 neurons,
# - ReLU activation,
# - an output layer with 7 classes.
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()

        # Linear layer: 7 inputs -> 32 hidden units
        self.fc1 = nn.Linear(7, 32)

        # Activation function
        self.relu = nn.ReLU()

        # Output layer: 32 hidden units -> 7 classes
        self.fc2 = nn.Linear(32, 7)

    def forward(self, x):
        # we Pass data through the first layer and activation
        x = self.fc1(x)
        x = self.relu(x)

        # then we Pass through the output layer
        x = self.fc2(x)
        return x

# Here we are Creating an instance of the model
model_cpu = MLP()

print("MLP has been successfully created .")
print(model_cpu)

MLP has been successfully created .
MLP(
  (fc1): Linear(in_features=7, out_features=32, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=32, out_features=7, bias=True)
)


Step 3: Training the MLP Model on the CPU

In thrid step, we will train MLP model using the CPU. The training process will follow the standard supervised learning flow. We will defin a loss function, will chose an optimizer and the will run 20 epochs and pring each of the stats to see the result of the model.



In [ ]:
import time
import torch.optim as optim

# we Use cross-entropy loss because this is a multi-class classification problem.
crt = nn.CrossEntropyLoss()

# we Use Adam optimizer, which is commonly used and works well for this kind of models.
opt = optim.Adam(model_cpu.parameters(), lr=0.001)

# Number of training epochs.
# We keep it small because the dataset is not very large.
epochs = 20

# Moving the model to CPU explicitly (even though it's already there).
device_cpu = torch.device("cpu")
model_cpu.to(device_cpu)

start_time = time.time() # We are marking time when training begins

# This is Training loop
for epoch in range(epochs):
    model_cpu.train()  # Set model to training mode
    running_loss = 0.0 # Start loss counter for eoch

    # Accuracy calculation
    correct = 0        # Count correct predictions
    total = 0          # Count total samples

    for batch_X, batch_y in train_ldr:
        # Moving data to CPU
        batch_X = batch_X.to(device_cpu)
        batch_y = batch_y.to(device_cpu)

        # Resetting gradients from the previous step
        opt.zero_grad()

        # Forward pass
        outputs = model_cpu(batch_X)

        # Computing loss
        loss = crt(outputs, batch_y)

        # Backward pass (computing gradients)
        loss.backward()

        # we update model parameters
        opt.step()

        running_loss += loss.item()

        # For accuracy calculation
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

    # Printing average loss for the epoch
    avg_loss = running_loss / len(train_ldr)

    # Printing Training accuracy and loss
    accuracy = correct / total
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Training Accuracy: {accuracy:.4f}")

    # Validation accuracy
    model_cpu.eval()  # Set model to evaluation mode
    val_correct = 0
    val_total = 0

    with torch.no_grad():  # No gradient calculation during validation
        for val_X, val_y in val_ldr:
            val_X = val_X.to(device_cpu)
            val_y = val_y.to(device_cpu)

            val_outputs = model_cpu(val_X)
            _, val_predicted = torch.max(val_outputs, 1)
            
            val_correct += (val_predicted == val_y).sum().item()
            val_total += val_y.size(0)

    val_accuracy = val_correct / val_total
    print(f"Validation Accuracy: {val_accuracy:.4f}")


end_time = time.time() # We are marking time when training ends
cpu_training_time = end_time - start_time

print("\nCPU training completed.")
print("Total CPU training time:", cpu_training_time, "seconds")

Epoch 1/20, Loss: 1.7271, Training Accuracy: 0.4100
Validation Accuracy: 0.4788
Epoch 2/20, Loss: 1.4473, Training Accuracy: 0.4906
Validation Accuracy: 0.4963
Epoch 3/20, Loss: 1.2933, Training Accuracy: 0.5078
Validation Accuracy: 0.5088
Epoch 4/20, Loss: 1.2215, Training Accuracy: 0.5088
Validation Accuracy: 0.5125
Epoch 5/20, Loss: 1.1867, Training Accuracy: 0.5144
Validation Accuracy: 0.5262
Epoch 6/20, Loss: 1.1662, Training Accuracy: 0.5169
Validation Accuracy: 0.5400
Epoch 7/20, Loss: 1.1521, Training Accuracy: 0.5278
Validation Accuracy: 0.5463
Epoch 8/20, Loss: 1.1412, Training Accuracy: 0.5262
Validation Accuracy: 0.5475
Epoch 9/20, Loss: 1.1328, Training Accuracy: 0.5281
Validation Accuracy: 0.5437
Epoch 10/20, Loss: 1.1257, Training Accuracy: 0.5284
Validation Accuracy: 0.5463
Epoch 11/20, Loss: 1.1203, Training Accuracy: 0.5322
Validation Accuracy: 0.5425
Epoch 12/20, Loss: 1.1149, Training Accuracy: 0.5284
Validation Accuracy: 0.5437
Epoch 13/20, Loss: 1.1103, Training A

Step 4: Training the MLP Model on the GPU

In this fourth step, we will use the same MLP model but will be trained using GPU. The training process is very much same as before.




In [4]:
import torch.optim as optim
import time

# we check if a GPU is available and selecting the device
device_gpu = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device_gpu)

# we Create a new model instance for GPU training
# We keep the architecture identical to the CPU model to ensure a fair comparison.
model_gpu = MLP().to(device_gpu)

# we define loss function and optimizer (same setup as CPU training)
crt = nn.CrossEntropyLoss()
opt = optim.Adam(model_gpu.parameters(), lr=0.001)

# epochs number and same as CPU
epochs = 20

start_time = time.time()  # Marking when GPU training begins

# This is Training loop on GPU
for epoch in range(epochs):
    model_gpu.train()
    running_loss = 0.0

    # we define counters for tracking training accuracy during each epoch
    correct = 0
    total = 0

    for batch_X, batch_y in train_ldr:
        # Moving the current batch to the GPU
        batch_X = batch_X.to(device_gpu)
        batch_y = batch_y.to(device_gpu)

        opt.zero_grad()

        # Forward pass through the network
        outputs = model_gpu(batch_X)

        # Loss calculation
        loss = crt(outputs, batch_y)

        # Backpropagation step
        loss.backward()

        # we update model parameters
        opt.step()

        running_loss += loss.item()

        # we update training accuracy counters
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

    # Average loss for the epoch
    avg_loss = running_loss / len(train_ldr)

    # we calculate training accuracy
    train_acc = correct / total
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Training Accuracy: {train_acc:.4f}")

    # Validation accuracy calculation
    model_gpu.eval()
    val_correct = 0
    val_total = 0

    # We evaluate the model on the validation set without gradient tracking.
    # This keeps the computation efficient and avoids unnecessary memory usage.
    with torch.no_grad():
        for val_X, val_y in val_ldr:
            val_X = val_X.to(device_gpu)
            val_y = val_y.to(device_gpu)

            val_outputs = model_gpu(val_X)
            _, val_predicted = torch.max(val_outputs, 1)

            val_correct += (val_predicted == val_y).sum().item()
            val_total += val_y.size(0)

    val_accuracy = val_correct / val_total
    print(f"Validation Accuracy: {val_accuracy:.4f}")


end_time = time.time()  # Marking when GPU training ends
gpu_training_time = end_time - start_time

print("\nGPU training completed.")
print("Total GPU training time:", gpu_training_time, "seconds")

Using device: cuda
Epoch 1/20, Loss: 1.7059, Training Accuracy: 0.3906
Validation Accuracy: 0.4775
Epoch 2/20, Loss: 1.4148, Training Accuracy: 0.4753
Validation Accuracy: 0.5000
Epoch 3/20, Loss: 1.2810, Training Accuracy: 0.4981
Validation Accuracy: 0.5112
Epoch 4/20, Loss: 1.2222, Training Accuracy: 0.5059
Validation Accuracy: 0.5275
Epoch 5/20, Loss: 1.1895, Training Accuracy: 0.5100
Validation Accuracy: 0.5363
Epoch 6/20, Loss: 1.1680, Training Accuracy: 0.5234
Validation Accuracy: 0.5350
Epoch 7/20, Loss: 1.1532, Training Accuracy: 0.5238
Validation Accuracy: 0.5387
Epoch 8/20, Loss: 1.1416, Training Accuracy: 0.5288
Validation Accuracy: 0.5400
Epoch 9/20, Loss: 1.1326, Training Accuracy: 0.5331
Validation Accuracy: 0.5563
Epoch 10/20, Loss: 1.1261, Training Accuracy: 0.5303
Validation Accuracy: 0.5600
Epoch 11/20, Loss: 1.1201, Training Accuracy: 0.5316
Validation Accuracy: 0.5513
Epoch 12/20, Loss: 1.1147, Training Accuracy: 0.5369
Validation Accuracy: 0.5625
Epoch 13/20, Loss:

Step 5: Evaluation of the MLP model

In this step, we are going to measure the performance of the MLP model on both CPU and GPU. This will contain two parts:
1. Measurement on validation set
2. Measuerment on inference time
Evaluating the first one will help us understand how well model can predecits classes and second one will help us know how fast the model can make choices.


In [5]:
import time
import torch

# We define a reusable helper function that evaluates the model by running
# it on the validation set and counting how many predictions are correct.
# The function works for both CPU and GPU depending on the 'device' argument.
def measure_accuracy(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():  # We disable gradient tracking since this is inference
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch) # Forward pass only
            _, predicted = torch.max(outputs, 1)

            total += y_batch.size(0) # Count how many samples we saw
            correct += (predicted == y_batch).sum().item()

    return correct / total



# CPU Evaluation

device_cpu = torch.device("cpu")
model_cpu.to(device_cpu)  # Move the trained CPU model to the CPU device

# We measure inference time by timing the accuracy computation.
start_cpu = time.time()
cpu_accuracy = measure_accuracy(model_cpu, val_ldr, device_cpu)
end_cpu = time.time()

cpu_inference_time = end_cpu - start_cpu

print("CPU Accuracy:", cpu_accuracy)
print("CPU Inference Time:", cpu_inference_time, "seconds")



# GPU Evaluation


device_gpu = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_gpu.to(device_gpu)  # Move the trained GPU model to the GPU device

# We run a short warm‑up pass to ensure the GPU is active.
# This avoids measuring any initial GPU startup overhead in our timing.
with torch.no_grad():
    for X_batch, _ in val_ldr:
        _ = model_gpu(X_batch.to(device_gpu))
        break

start_gpu = time.time()
gpu_accuracy = measure_accuracy(model_gpu, val_ldr, device_gpu)
end_gpu = time.time()

gpu_inference_time = end_gpu - start_gpu

print("\nGPU Accuracy:", gpu_accuracy)
print("GPU Inference Time:", gpu_inference_time, "seconds")

CPU Accuracy: 0.555
CPU Inference Time: 0.011709451675415039 seconds

GPU Accuracy: 0.56625
GPU Inference Time: 0.008978605270385742 seconds


Step 6: Performance comparison of CPU vs GPU

In the final step, we will make a summary and will compare the result of MLP model on both CPU and GPU. This will help us know the fair comparison between these resources (CPU vs GPU) using the same model.


In [6]:
import pandas as pd

comp_data = {
    "Parameter": ["Training Time (s)", "Inference Time (s)", "Validation Accuracy"],
    "CPU": [cpu_training_time, cpu_inference_time, cpu_accuracy],
    "GPU": [gpu_training_time, gpu_inference_time, gpu_accuracy]
}

# Convert the dictionary of metrics into a structured pandas DataFrame,
# making it easy to display the CPU vs GPU results in a clean table format.
comp_df = pd.DataFrame(comp_data)

print("CPU vs GPU Performance Comparison:\n")
print(comp_df)

CPU vs GPU Performance Comparison:

             Parameter       CPU       GPU
0    Training Time (s)  1.926447  2.683367
1   Inference Time (s)  0.011709  0.008979
2  Validation Accuracy  0.555000  0.566250
